In [ ]:
import time
import random
from datetime import datetime

# Each staff member: name, phone (E.164 format), skill, distance from shop (km), past acceptance rate (0-1)
roster = [
    {"name": "Sunethra",  "phone": "+91XXXXXXXX01", "skill": "cashier", "distance_km": 1.2, "acceptance_rate": 0.92},
    {"name": "Vishal", "phone": "+91XXXXXXXX02", "skill": "cook",    "distance_km": 3.5, "acceptance_rate": 0.80},
    {"name": "Rithanya",  "phone": "+91XXXXXXXX03", "skill": "cashier", "distance_km": 0.8, "acceptance_rate": 0.65},
    {"name": "Vinitha", "phone": "+91XXXXXXXX04", "skill": "server",  "distance_km": 5.0, "acceptance_rate": 0.88},
    {"name": "Kavya",  "phone": "+91XXXXXXXX05", "skill": "cashier", "distance_km": 2.1, "acceptance_rate": 0.70},
    {"name": "Deepak", "phone": "+91XXXXXXXX06", "skill": "cook",    "distance_km": 1.9, "acceptance_rate": 0.55},
]

for s in roster:
    print(f"{s['name']:8s} | skill={s['skill']:8s} | {s['distance_km']:>4}km | accept_rate={s['acceptance_rate']:.0%}")

Sunethra | skill=cashier  |  1.2km | accept_rate=92%
Vishal   | skill=cook     |  3.5km | accept_rate=80%
Rithanya | skill=cashier  |  0.8km | accept_rate=65%
Vinitha  | skill=server   |  5.0km | accept_rate=88%
Kavya    | skill=cashier  |  2.1km | accept_rate=70%
Deepak   | skill=cook     |  1.9km | accept_rate=55%


In [ ]:
def estimate_eta_minutes(distance_km, speed_kmph=20):
    """Rough ETA assuming average bike/scooter speed for a local backup worker."""
    return round((distance_km / speed_kmph) * 60, 1)


def score_candidate(candidate, needed_skill, weights=(0.4, 0.3, 0.3)):
    """Higher score = better replacement candidate.
    weights = (skill_weight, proximity_weight, acceptance_weight)
    """
    w_skill, w_dist, w_accept = weights
    skill_score = 1.0 if candidate["skill"] == needed_skill else 0.3  # partial credit for cross-trained staff
    proximity_score = max(0, 1 - candidate["distance_km"] / 10)
    accept_score = candidate["acceptance_rate"]
    total = (w_skill * skill_score) + (w_dist * proximity_score) + (w_accept * accept_score)
    return round(total, 3)


def explain_candidate(candidate, needed_skill):
    skill_note = "exact skill match" if candidate["skill"] == needed_skill else f"cross-trained ({candidate['skill']})"
    eta = estimate_eta_minutes(candidate["distance_km"])
    return f"{skill_note}, ~{eta} min ETA, {candidate['acceptance_rate']:.0%} historical reliability"


def rank_candidates(roster, needed_skill, exclude_names=None):
    exclude_names = exclude_names or set()
    available = [c for c in roster if c["name"] not in exclude_names]
    ranked = sorted(available, key=lambda c: score_candidate(c, needed_skill), reverse=True)
    for c in ranked:
        c["_score"] = score_candidate(c, needed_skill)
        c["_eta"] = estimate_eta_minutes(c["distance_km"])
    return ranked


# Demo: shop needs a cashier right now
ranked = rank_candidates(roster, needed_skill="cashier")
print("Ranked backup candidates for role: cashier\n")
for i, c in enumerate(ranked, 1):
    print(f"{i}. {c['name']:8s} score={c['_score']:.2f}  — {explain_candidate(c, 'cashier')}")

Ranked backup candidates for role: cashier

1. Sunethra score=0.94  — exact skill match, ~3.6 min ETA, 92% historical reliability
2. Rithanya score=0.87  — exact skill match, ~2.4 min ETA, 65% historical reliability
3. Kavya    score=0.85  — exact skill match, ~6.3 min ETA, 70% historical reliability
4. Vishal   score=0.56  — cross-trained (cook), ~10.5 min ETA, 80% historical reliability
5. Vinitha  score=0.53  — cross-trained (server), ~15.0 min ETA, 88% historical reliability
6. Deepak   score=0.53  — cross-trained (cook), ~5.7 min ETA, 55% historical reliability


In [ ]:
MODE = "simulate"   # change to "real" once you have Twilio credentials set up

# --- Twilio setup (only used when MODE == "real") ---
TWILIO_ACCOUNT_SID = "PASTE_YOUR_SID"
TWILIO_AUTH_TOKEN  = "PASTE_YOUR_AUTH_TOKEN"
TWILIO_FROM_NUMBER = "+1XXXXXXXXXX"

if MODE == "real":
    from twilio.rest import Client
    twilio_client = Client(TWILIO_ACCOUNT_SID, TWILIO_AUTH_TOKEN)


def place_call(candidate):
    """Attempt to call the candidate. Returns 'answered' or 'no_answer'."""
    if MODE == "simulate":
        time.sleep(0.5)
        return random.choices(["answered", "no_answer"], weights=[0.4, 0.6])[0]
    else:
        call = twilio_client.calls.create(
            to=candidate["phone"],
            from_=TWILIO_FROM_NUMBER,
            url="https://YOUR_NGROK_URL/voice-response",
        )
        return "call_placed"


def send_sms(candidate, message):
    """Fallback SMS if the call goes unanswered."""
    if MODE == "simulate":
        time.sleep(0.3)
        return random.choices(["accepted", "declined", "no_response"], weights=[0.5, 0.2, 0.3])[0]
    else:
        sms = twilio_client.messages.create(
            to=candidate["phone"],
            from_=TWILIO_FROM_NUMBER,
            body=message,
        )
        return "sms_sent"


def get_call_response(candidate):
    """In simulate mode, mimics whether the answered call resulted in accept/decline."""
    if MODE == "simulate":
        return random.choices(["accepted", "declined"], weights=[0.6, 0.4])[0]
    else:
        
        return "pending_webhook"

In [ ]:
def run_panic_chain(roster, needed_skill, max_minutes=5, sms_wait_seconds=1.0, exclude_names=None):
    """Runs the escalation chain for one role and prints a live status feed."""
    exclude_names = exclude_names or set()
    start = time.time()
    ranked = rank_candidates(roster, needed_skill, exclude_names)
    log = []

    def feed(msg):
        elapsed = time.time() - start
        line = f"[{elapsed:5.1f}s] {msg}"
        print(line)
        log.append(line)

    feed(f"🔴 Escalation started for role: {needed_skill}")

    for candidate in ranked:
        if (time.time() - start) / 60 > max_minutes:
            feed(f"⏰ {needed_skill}: exceeded 5-min target — flagging for manual owner call.")
            return None, log

        feed(f"📞 Calling {candidate['name']} — {explain_candidate(candidate, needed_skill)}")
        call_result = place_call(candidate)

        if call_result == "answered":
            response = get_call_response(candidate)
            if response == "accepted":
                feed(f"✅ {candidate['name']} ACCEPTED (call). Confirmed in {time.time()-start:.1f}s.")
                return candidate, log
            feed(f"❌ {candidate['name']} declined. Escalating...")
            continue

        feed(f"📵 No answer from {candidate['name']}. SMS fallback sent...")
        time.sleep(sms_wait_seconds)
        sms_result = send_sms(candidate, "URGENT: cover a shift now? Reply YES/NO.")
        if sms_result == "accepted":
            feed(f"✅ {candidate['name']} ACCEPTED (SMS). Confirmed in {time.time()-start:.1f}s.")
            return candidate, log
        feed(f"❌ No acceptance ({sms_result}). Escalating...")

    feed(f"🚨 Roster exhausted for {needed_skill} — no replacement found.")
    return None, log


def run_multi_role_panic(roster, needed_skills):
    """Handles multiple simultaneous no-shows without double-booking the same backup."""
    results = {}
    assigned = set()
    print(f"🔴🔴 PANIC — {len(needed_skills)} staff missing: {needed_skills}\n")
    for role in needed_skills:
        winner, log = run_panic_chain(roster, needed_skill=role, exclude_names=assigned)
        results[role] = winner
        if winner:
            assigned.add(winner["name"])
        print()
    return results


# Demo: TWO staff missing at once — cashier and cook
results = run_multi_role_panic(roster, needed_skills=["cashier", "cook"])

🔴🔴 PANIC — 2 staff missing: ['cashier', 'cook']

[  0.0s] 🔴 Escalation started for role: cashier
[  0.0s] 📞 Calling Sunethra — exact skill match, ~3.6 min ETA, 92% historical reliability
[  0.5s] 📵 No answer from Sunethra. SMS fallback sent...
[  1.8s] ❌ No acceptance (declined). Escalating...
[  1.8s] 📞 Calling Rithanya — exact skill match, ~2.4 min ETA, 65% historical reliability
[  2.3s] 📵 No answer from Rithanya. SMS fallback sent...
[  3.6s] ❌ No acceptance (no_response). Escalating...
[  3.6s] 📞 Calling Kavya — exact skill match, ~6.3 min ETA, 70% historical reliability
[  4.1s] ✅ Kavya ACCEPTED (call). Confirmed in 4.1s.

[  0.0s] 🔴 Escalation started for role: cook
[  0.0s] 📞 Calling Vishal — exact skill match, ~10.5 min ETA, 80% historical reliability
[  0.5s] 📵 No answer from Vishal. SMS fallback sent...
[  1.8s] ✅ Vishal ACCEPTED (SMS). Confirmed in 1.8s.



In [ ]:
def generate_owner_report(results):
    print("=" * 50)
    print("📋 SHIFT COVERAGE REPORT")
    print("=" * 50)
    filled, unfilled = 0, 0
    for role, winner in results.items():
        if winner:
            filled += 1
            print(f"✅ {role.upper()}: covered by {winner['name']} (ETA ~{winner['_eta']} min)")
        else:
            unfilled += 1
            print(f"⚠️  {role.upper()}: NOT covered — call remaining staff manually or reduce hours.")
    print("-" * 50)
    print(f"Summary: {filled}/{filled+unfilled} roles covered automatically.")


generate_owner_report(results)

📋 SHIFT COVERAGE REPORT
✅ CASHIER: covered by Kavya (ETA ~6.3 min)
✅ COOK: covered by Vishal (ETA ~10.5 min)
--------------------------------------------------
Summary: 2/2 roles covered automatically.


In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    skill_select = widgets.SelectMultiple(
        options=sorted(set(s["skill"] for s in roster)),
        description="Roles missing:",
        rows=4,
    )
    panic_button = widgets.Button(
        description="🔴 PANIC — FIND BACKUPS NOW",
        button_style="danger",
        layout=widgets.Layout(width="320px", height="50px"),
    )
    output = widgets.Output()

    def on_click(b):
        with output:
            clear_output()
            if not skill_select.value:
                print("Select at least one missing role first.")
                return
            res = run_multi_role_panic(roster, needed_skills=list(skill_select.value))
            generate_owner_report(res)

    panic_button.on_click(on_click)
    display(skill_select, panic_button, output)

except ImportError:
    print("ipywidgets not installed — run: pip install ipywidgets")
    res = run_multi_role_panic(roster, needed_skills=["cashier", "cook"])
    generate_owner_report(res)

ipywidgets not installed — run: pip install ipywidgets
🔴🔴 PANIC — 2 staff missing: ['cashier', 'cook']

[  0.0s] 🔴 Escalation started for role: cashier
[  0.0s] 📞 Calling Sunethra — exact skill match, ~3.6 min ETA, 92% historical reliability


[  0.5s] ✅ Sunethra ACCEPTED (call). Confirmed in 0.5s.

[  0.0s] 🔴 Escalation started for role: cook
[  0.0s] 📞 Calling Vishal — exact skill match, ~10.5 min ETA, 80% historical reliability
[  0.5s] ✅ Vishal ACCEPTED (call). Confirmed in 0.5s.

📋 SHIFT COVERAGE REPORT
✅ CASHIER: covered by Sunethra (ETA ~3.6 min)
✅ COOK: covered by Vishal (ETA ~10.5 min)
--------------------------------------------------
Summary: 2/2 roles covered automatically.
